<p align="center">
  <a href="https://colab.research.google.com/github/samuel-c-santos/opensr-sentinel-pipeline/blob/main/OpenSR_Producao_Poligono_Colab.ipynb" target="_blank">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir no Google Colab" width="180">
  </a>
</p>

# ESA OpenSR - Super-Resolução de Sentinel-2 por Polígono (AOI)
## Processamento em Lote com Janela Deslizante (10m para 2.5m / 4x)

Este notebook executa o fluxo operacional de super-resolucao espacial do sensor Sentinel-2 sobre Áreas de Interesse (AOI) delimitadas por polígonos (Shapefile ou GeoJSON), com exportação direta para o Google Drive.

### Características do Fluxo:
- **Delimitação por Vetor**: Suporta upload de Shapefile (.zip) ou polígono interativo.
- **Processamento em Janela Deslizante (Tiling)**: Divide a área em blocos regulares com sobreposição (overlap) para garantir estabilidade de memória de vídeo na GPU e eliminar descontinuidades nas emendas.
- **Preservação de Metadados**: O arquivo GeoTIFF final mantém o sistema de referência de coordenadas (CRS), a geotransformação ajustada para 2.5m/pixel e as bandas multiespectrais VNIR (B04, B03, B02 e B08).
- **Armazenamento Integrado**: Conexão com o Google Drive para gravação dos resultados finais.


## 1. Instalação das Dependências

> **Importante:** Certifique-se de ativar a GPU no menu: *Ambiente de Execução > Alterar tipo de ambiente de execução > GPU (T4)*.

In [ ]:
# Instalação dos pacotes necessários
!pip install -q opensr-model "opensr-utils[model]" omegaconf rasterio geopandas geemap earthengine-api

## 2. Autenticação e Inicialização do Google Earth Engine e Google Drive
Utiliza o projeto configurado `sentinel-landsat-download` e monta o Google Drive em `/content/drive`.

In [ ]:
import ee
import os
from google.colab import drive

# 1. Inicialização do GEE
GEE_PROJECT = "sentinel-landsat-download"
try:
    ee.Initialize(project=GEE_PROJECT)
    print(f"✅ Google Earth Engine inicializado no projeto: {GEE_PROJECT}")
except Exception as e:
    ee.Authenticate()
    ee.Initialize(project=GEE_PROJECT)
    print(f"✅ Google Earth Engine autenticado e inicializado: {GEE_PROJECT}")

# 2. Montagem do Google Drive
drive.mount("/content/drive")

# 3. Criação de pastas de trabalho
WORK_DIR = "/content/data"
OUTPUT_LOCAL_DIR = "/content/output"
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/OpenSR_Outputs"

os.makedirs(WORK_DIR, exist_ok=True)
os.makedirs(OUTPUT_LOCAL_DIR, exist_ok=True)
os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)

print(f"📁 Pasta de saída no Drive configurada: {DRIVE_OUTPUT_DIR}")

## 3. Definição da Área de Interesse (AOI)
Você pode fazer upload de um arquivo `.zip` contendo o Shapefile (`.shp`, `.shx`, `.dbf`, `.prj`) ou utilizar o polígono de referência padrão (região de Belém/PA).

In [ ]:
import zipfile
import glob
import geopandas as gpd
from google.colab import files

# Defina True para fazer upload de um Shapefile (.zip), ou False para usar o polígono de teste
FAZER_UPLOAD_SHAPEFILE = False

if FAZER_UPLOAD_SHAPEFILE:
    print("Selecione o arquivo .zip contendo os componentes do Shapefile...")
    uploaded = files.upload()
    zip_name = list(uploaded.keys())[0]
    with zipfile.ZipFile(zip_name, 'r') as zip_ref:
        zip_ref.extractall(WORK_DIR)
    shp_candidates = glob.glob(os.path.join(WORK_DIR, "*.shp"))
    if not shp_candidates:
        raise FileNotFoundError("Nenhum arquivo .shp encontrado dentro do .zip enviado.")
    gdf = gpd.read_file(shp_candidates[0])
    if gdf.crs != "EPSG:4326":
        gdf = gdf.to_crs("EPSG:4326")
    aoi_geometry = ee.Geometry(gdf.unary_union.__geo_interface__)
    print(f"✅ Shapefile carregado com sucesso: {shp_candidates[0]}")
else:
    # Polígono padrão de teste (Área do IDEFLOR-Bio / Utinga em Belém/PA)
    coords = [
        [-48.4550, -1.4350],
        [-48.4350, -1.4350],
        [-48.4350, -1.4150],
        [-48.4550, -1.4150],
        [-48.4550, -1.4350]
    ]
    aoi_geometry = ee.Geometry.Polygon([coords])
    print("✅ Utilizando polígono de referência padrão (Belém/PA).")

aoi_bounds = aoi_geometry.bounds()
print("Limites da AOI definidos com sucesso.")

## 4. Seleção e Download do Mosaico/Cena Sentinel-2 (10m)
Filtra a coleção de superfície `COPERNICUS/S2_SR_HARMONIZED` e faz o download do GeoTIFF de 4 bandas (B04, B03, B02 e B08) na resolução nativa de 10m.

In [ ]:
import urllib.request
import rasterio

START_DATE = "2025-08-01"
END_DATE = "2026-03-01"
MAX_CLOUD = 25

collection = (
    ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
    .filterBounds(aoi_geometry)
    .filterDate(START_DATE, END_DATE)
    .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", MAX_CLOUD))
    .sort("CLOUDY_PIXEL_PERCENTAGE")
)

n_imgs = collection.size().getInfo()
if n_imgs == 0:
    print("Nenhuma cena limpa no período estrito. Selecionando a melhor disponível no ano...")
    collection = (
        ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
        .filterBounds(aoi_geometry)
        .sort("CLOUDY_PIXEL_PERCENTAGE")
    )
    s2_img = ee.Image(collection.first())
else:
    s2_img = ee.Image(collection.first())
    print(f"Encontradas {n_imgs} cenas elegíveis. Selecionando a com menor cobertura de nuvens.")

img_date = s2_img.date().format("YYYY-MM-dd").getInfo()
img_id = s2_img.get("system:index").getInfo()
print(f"📅 Data da cena selecionada: {img_date}")
print(f"🏷️ ID da cena: {img_id}")

# Seleção das 4 bandas VNIR de 10m
s2_bands = s2_img.select(["B4", "B3", "B2", "B8"]).clip(aoi_bounds)

input_tif_path = os.path.join(WORK_DIR, "aoi_s2_input_10m.tif")
download_url = s2_bands.getDownloadURL({
    "scale": 10,
    "crs": "EPSG:3857",
    "region": aoi_bounds,
    "format": "GEO_TIFF"
})

print("📥 Baixando imagem Sentinel-2 (10m) para a AOI...")
urllib.request.urlretrieve(download_url, input_tif_path)

with rasterio.open(input_tif_path) as src:
    lr_shape = src.shape
    lr_count = src.count
    lr_profile = src.profile
    print(f"✅ Imagem Sentinel-2 baixada com sucesso!")
    print(f"📐 Dimensões LR (10m): {lr_count} bandas, {lr_shape[0]}x{lr_shape[1]} pixels")

## 5. Carregamento do Modelo LDSR-S2 (ESA OpenSR)
Inicializa os pesos do modelo de difusão latente treinados especificamente para Sentinel-2.

In [ ]:
from io import StringIO
import requests
import torch
from omegaconf import OmegaConf
import opensr_model

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🚀 Dispositivo selecionado: {device.upper()}")
if device == "cpu":
    print("⚠️ ATENÇÃO: Recomenda-se ativar a GPU T4 para processamento de polígonos com janelas deslizantes.")

config_url = "https://raw.githubusercontent.com/ESAOpenSR/opensr-model/refs/heads/main/opensr_model/configs/config_10m.yaml"
response = requests.get(config_url)
config = OmegaConf.load(StringIO(response.text))

print("⏳ Carregando modelo LDSR-S2 (HuggingFace)...")
model = opensr_model.SRLatentDiffusion(config, device=device)
model.load_pretrained(config.ckpt_version)
model.eval()
print("✅ Modelo pronto para inferência por janelas deslizantes!")

## 6. Processamento em Janela Deslizante (Tiling com Overlap)
A imagem é subdividida em blocos de 128x128 pixels de baixa resolução com sobreposição de 16 pixels. Cada bloco é processado individualmente pela rede neural e reconstituído com ponderação de bordas para evitar linhas de emenda aparentes.

In [ ]:
import numpy as np
import math

def process_large_raster_tiled(input_path, model, device, window_size=128, overlap=16, scale=4):
    with rasterio.open(input_path) as src:
        img = src.read()  # (C, H, W)
        profile = src.profile

    c, h, w = img.shape
    sr_h, sr_w = h * scale, w * scale

    # Acumuladores para ponderação suave de janelas sobrepostas
    output = np.zeros((c, sr_h, sr_w), dtype=np.float32)
    weight = np.zeros((sr_h, sr_w), dtype=np.float32)

    step = window_size - overlap
    y_steps = max(1, math.ceil((h - window_size) / step) + 1) if h > window_size else 1
    x_steps = max(1, math.ceil((w - window_size) / step) + 1) if w > window_size else 1

    print(f"Iniciando processamento de grade: {y_steps} x {x_steps} blocos...")

    # Máscara de peso 2D (janela hanning para fusão sem costuras)
    win_sr = window_size * scale
    han_1d = np.hanning(win_sr)
    win_weight = np.outer(han_1d, han_1d).astype(np.float32)
    win_weight = np.clip(win_weight, 0.05, 1.0)  # Evita zeros absolutos nas bordas

    count = 0
    total_tiles = y_steps * x_steps

    for yi in range(y_steps):
        y1 = min(yi * step, max(0, h - window_size))
        y2 = min(y1 + window_size, h)
        for xi in range(x_steps):
            x1 = min(xi * step, max(0, w - window_size))
            x2 = min(x1 + window_size, w)

            # Patch de entrada (4, window_size, window_size)
            patch = img[:, y1:y2, x1:x2].astype(np.float32)
            patch_h, patch_w = patch.shape[1], patch.shape[2]

            # Normalização (0-10000 para 0-1)
            patch_tensor = torch.from_numpy(patch).float().unsqueeze(0) / 10000.0
            patch_tensor = patch_tensor.to(device)

            with torch.no_grad():
                sr_patch = model(patch_tensor)

            sr_patch_np = sr_patch.squeeze(0).cpu().numpy()

            # Coordenadas na imagem de saída
            sy1, sy2 = y1 * scale, y2 * scale
            sx1, sx2 = x1 * scale, x2 * scale
            cur_win_sr_h, cur_win_sr_w = sy2 - sy1, sx2 - sx1

            cur_weight = win_weight[:cur_win_sr_h, :cur_win_sr_w]

            for band in range(c):
                output[band, sy1:sy2, sx1:sx2] += sr_patch_np[band, :cur_win_sr_h, :cur_win_sr_w] * cur_weight
            weight[sy1:sy2, sx1:sx2] += cur_weight

            count += 1
            if count % 5 == 0 or count == total_tiles:
                print(f"Progresso: Bloco {count}/{total_tiles} processado.")

    # Normalização final pela soma dos pesos
    weight = np.maximum(weight, 1e-6)
    for band in range(c):
        output[band] /= weight

    output = np.clip(output * 10000.0, 0, 10000).astype(np.uint16)
    return output, profile

sr_result, in_profile = process_large_raster_tiled(input_tif_path, model, device)
print("✨ Processamento por janelas deslizantes concluído!")
print(f"📐 Dimensões finais da cena super-resolvida (2.5m): {sr_result.shape[0]} bandas, {sr_result.shape[1]}x{sr_result.shape[2]} pixels")

## 7. Gravação e Exportação do GeoTIFF de 2.5m para o Google Drive
Atualiza a resolução e a matriz de transformação afim, gravando o produto final tanto localmente quanto na pasta sincronizada do Google Drive.

In [ ]:
import shutil
from rasterio.transform import Affine

scale_factor = 4
orig_transform = in_profile["transform"]
new_transform = Affine(
    orig_transform.a / scale_factor, orig_transform.b, orig_transform.c,
    orig_transform.d, orig_transform.e / scale_factor, orig_transform.f
)

out_profile = in_profile.copy()
out_profile.update({
    "height": sr_result.shape[1],
    "width": sr_result.shape[2],
    "transform": new_transform,
    "dtype": "uint16",
    "nodata": 0
})

output_local_file = os.path.join(OUTPUT_LOCAL_DIR, f"opensr_2_5m_{img_date}.tif")
output_drive_file = os.path.join(DRIVE_OUTPUT_DIR, f"opensr_2_5m_{img_date}.tif")

# 1. Grava no disco local da sessão
with rasterio.open(output_local_file, "w", **out_profile) as dst:
    dst.write(sr_result)
print(f"💾 Arquivo local gravado: {output_local_file}")

# 2. Copia para o Google Drive
shutil.copy2(output_local_file, output_drive_file)
print(f"✅ GeoTIFF de 2.5m exportado para o Google Drive com sucesso:")
print(f"   📍 {output_drive_file}")

## 8. Inspecção Visual Comparativa da AOI (10m vs 2.5m)

In [ ]:
import matplotlib.pyplot as plt

def to_rgb(img_bands, r_idx, g_idx, b_idx, p_min=2, p_max=98):
    rgb = np.stack([img_bands[r_idx], img_bands[g_idx], img_bands[b_idx]], axis=-1).astype(float)
    for c in range(3):
        band = rgb[..., c]
        v_min, v_max = np.percentile(band, (p_min, p_max))
        if v_max > v_min:
            rgb[..., c] = np.clip((band - v_min) / (v_max - v_min), 0, 1)
    return rgb

with rasterio.open(input_tif_path) as src:
    orig_lr = src.read()

rgb_lr = to_rgb(orig_lr, 0, 1, 2)  # B4, B3, B2
rgb_sr = to_rgb(sr_result, 0, 1, 2)

nir_lr = to_rgb(orig_lr, 3, 0, 1)  # B8, B4, B3 (Falsa Cor)
nir_sr = to_rgb(sr_result, 3, 0, 1)

fig, axes = plt.subplots(2, 2, figsize=(14, 14))

axes[0, 0].imshow(rgb_lr)
axes[0, 0].set_title(f"Entrada Sentinel-2 Original (10m/px) - RGB\nData: {img_date}", fontsize=11, fontweight="bold")
axes[0, 0].axis("off")

axes[0, 1].imshow(rgb_sr)
axes[0, 1].set_title("Super-Resolucao ESA OpenSR (2.5m/px) - RGB\nReconstruido via Janelas Deslizantes", fontsize=11, fontweight="bold", color="darkgreen")
axes[0, 1].axis("off")

axes[1, 0].imshow(nir_lr)
axes[1, 0].set_title("Entrada Sentinel-2 Original (10m/px) - Falsa Cor (NIR)", fontsize=11, fontweight="bold")
axes[1, 0].axis("off")

axes[1, 1].imshow(nir_sr)
axes[1, 1].set_title("Super-Resolucao ESA OpenSR (2.5m/px) - Falsa Cor (NIR)", fontsize=11, fontweight="bold", color="darkgreen")
axes[1, 1].axis("off")

plt.tight_layout()
plt.show()